# Overview

In this notebook, we learned how to use **Hugging Face Pipelines** to perform various **Natural Language Processing (NLP)** and **Generative AI** tasks using pre-trained models. Instead of training AI models from scratch, we downloaded and used models available on the **Hugging Face Hub**. The notebook demonstrated how pipelines simplify AI development by automatically handling model loading, tokenization, inference, and output generation.

## What We Learned

- Installed the required libraries such as **Transformers** and other dependencies.
- Verified the availability of a **T4 GPU** for faster model inference.
- Created a **Hugging Face** account and generated an **Access Token**.
- Stored the access token securely in **Google Colab Secrets** and authenticated with Hugging Face.
- Learned the difference between **Training** and **Inference**, focusing only on inference using pre-trained models.
- Understood what a **Pipeline** is and how it simplifies working with AI models.
- Performed **Sentiment Analysis** to identify whether text is positive or negative.
- Used **Named Entity Recognition (NER)** to identify entities such as people, organizations, and locations.
- Implemented **Question Answering** to extract answers from a given context.
- Generated concise summaries from long text using **Text Summarization**.
- Translated text between different languages using **Translation Pipelines**.
- Used **Zero-Shot Classification** to classify text into custom categories without additional training.
- Generated text from prompts using a **Text Generation** model.
- Generated images from text descriptions using the **Stable Diffusion** Text-to-Image pipeline.
- Converted text into natural-sounding speech using the **SpeechT5** Text-to-Speech pipeline.
- Explored the different pipeline tasks available in the Hugging Face Transformers library.

## Outcome

By the end of this notebook, we understood how Hugging Face Pipelines provide a simple and efficient way to use powerful pre-trained AI models for both NLP and Generative AI tasks. We learned how to authenticate with Hugging Face, create pipelines for different tasks, and generate text, images, speech, translations, summaries, and classifications with only a few lines of code.

In [ ]:

!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

In [ ]:
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device="cuda")
result = my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:

better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
ner = pipeline("ner", device="cuda")
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in result:
  print(entity)

In [ ]:
question="What are Hugging Face pipelines?"
context="Pipelines are a high level API for inference of LLMs with common tasks"

question_answerer = pipeline("question-answering", device="cuda")
result = question_answerer(question=question, context=context)
print(result)

In [ ]:

summarizer = pipeline("summarization", device="cuda")
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

In [ ]:

translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:

classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

In [ ]:

generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])